# Load labor_cost directly from GitLab

This uses your colleague's runtime-handle approach: download one commit,
load its package in the running kernel, then use `lc = CODE.package`.
**No pip installation, installed `labor-cost-model` metadata, or kernel
restart is needed to load the library.** Keep `CODE` alive while working.

First prepare dependencies through your company's existing procedure using
[runtime/requirements-reporting.txt](../runtime/requirements-reporting.txt),
or the core `requirements.txt` with `enable_reporting = False` below.
Select that environment's kernel. The loader does not install dependencies;
restart the kernel if you changed those dependencies. Python 3.10+ is required,
with a version supported by the pinned dependency stack.

The bootstrap and package must come from your trusted repository. Every request
uses the same immutable commit after resolving the requested ref. The token
remains in the caller-owned requests session, which is cleared and closed.
See the [deployment guide](../docs/gitlab-deployment.md).

In [ ]:
import sys

GITLAB_API = "https://gitlab.tcsbank.ru/api/v4"
GITLAB_PROJECT = 278363
GITLAB_REF = "master"  # Use a release tag/full SHA for repeat runs.
PACKAGE_SUBDIR = "production"  # Use "." if production/ is the repo root.
enable_reporting = True
renderer = "plotly_mimetype"  # Classic Notebook: "notebook".

# Z_ENV_GITLAB_TOKEN is supplied by your existing secret mechanism.
print("Notebook Python:", sys.executable)
print("Python version:", sys.version.split()[0])

## Fetch and load one version

Run this cell once per working session. If it has already loaded an open
runtime, it asks you to finish and close that runtime before replacing `CODE`.
Use the cleanup example at the end. After loading another commit, recreate
models, datasets and results through the new `lc`; do not mix runtime objects.

In [ ]:
import json
import os
import re
from urllib.parse import quote, urlsplit

import requests


def gitlab_read(session, endpoint, **params):
    """Read this GitLab project without redirects or credential logging."""
    url = (
        GITLAB_API.rstrip("/") + "/projects/"
        + quote(str(GITLAB_PROJECT), safe="") + "/" + endpoint
    )
    try:
        response = session.get(
            url, params=params, timeout=30, allow_redirects=False,
        )
    except requests.RequestException:
        raise RuntimeError(
            "GitLab request failed; check network and corporate CA setup."
        ) from None
    if response.status_code != 200:
        status = response.status_code
        response.close()
        raise RuntimeError(
            f"GitLab returned HTTP {status}; check URL, project, ref and "
            "token permissions. Redirects are not followed."
        )
    return response


existing_runtime = globals().get("CODE")
if existing_runtime is not None and not existing_runtime.closed:
    raise RuntimeError("Finish your analysis and close CODE before reloading.")
parsed_api = urlsplit(GITLAB_API)
if (
    parsed_api.scheme != "https" or not parsed_api.hostname
    or parsed_api.username is not None or parsed_api.password is not None
    or parsed_api.query or parsed_api.fragment
):
    raise ValueError("Use an HTTPS API URL without credentials or query.")
if sys.version_info < (3, 10):
    raise RuntimeError("Select a Python 3.10+ kernel before loading.")

prefix = "" if PACKAGE_SUBDIR == "." else PACKAGE_SUBDIR.strip("/") + "/"
bootstrap_path = prefix + "runtime/bootstrap.py"
package_path = prefix + "src/labor_cost"
with requests.Session() as gitlab_session:
    gitlab_session.headers["PRIVATE-TOKEN"] = (
        globals().get("Z_ENV_GITLAB_TOKEN")
        or os.environ.get("Z_ENV_GITLAB_TOKEN")
    )
    try:
        if not gitlab_session.headers["PRIVATE-TOKEN"]:
            raise RuntimeError("Provide Z_ENV_GITLAB_TOKEN via secrets.")
        with gitlab_read(
            gitlab_session,
            "repository/commits/" + quote(GITLAB_REF, safe=""),
        ) as response:
            CODE_COMMIT = response.json()["id"]
        if not re.fullmatch(r"[0-9a-fA-F]{40}|[0-9a-fA-F]{64}", CODE_COMMIT):
            raise RuntimeError("GitLab did not return a full commit SHA.")
        with gitlab_read(
            gitlab_session,
            "repository/files/" + quote(bootstrap_path, safe="") + "/raw",
            ref=CODE_COMMIT,
        ) as response:
            loader_source = response.text
        loader = {"__name__": "labor_cost_gitlab_bootstrap"}
        exec(compile(loader_source, "GitLab/runtime/bootstrap.py", "exec"),
             loader)
        CODE = loader["load_project"](
            GITLAB_API, GITLAB_PROJECT, CODE_COMMIT, gitlab_session,
            ref=GITLAB_REF, package_path=package_path,
        )
    finally:
        gitlab_session.headers.clear()

lc = CODE.package
print("Requested ref:", GITLAB_REF)
print("Resolved commit:", CODE.provenance["commit"])
print("Package version:", lc.__version__)
print("Loaded source:", lc.__file__)
print("Ready in this kernel; no restart required.")

## Use the public API through `lc`

Every public symbol is an attribute: `lc.ModelSpec`, `lc.DatasetSnapshot`,
`lc.compile_model`, `lc.analyze`, `lc.report`, and so on. Do **not** use
`from labor_cost import ...`: it selects a separate installed package (or
fails if none is installed), not this loaded copy. `lc` is a module object,
not an importable alias; use `lc.analyze(...)`, not `from lc import analyze`.

For an existing notebook, replace its library import block with explicit
bindings if you prefer to keep unqualified calls:

```python
AnalysisRequest = lc.AnalysisRequest
DatasetSnapshot = lc.DatasetSnapshot
analyze = lc.analyze
```

For module-level access, `CODE.import_module("api")` matches the colleague's
pattern. The package handle is preferable because it exposes the whole public
API, including authoring and reporting.

In [ ]:
api = CODE.import_module("api")
assert api.analyze is lc.analyze
print("Public API:", lc.__name__)

## Compile, preview, cost and analyze

This small signed-payment model verifies deployment; it does not replace your
payroll model. Expected costs are **−100 → 150**, a reconciled **+250** change.
Source payroll still comes from your separately supplied `DatasetSnapshot`.
The source loader does not retrieve payroll or create a model catalog.

In [ ]:
import pandas as pd
from IPython.display import display

model = lc.ModelSpec(
    "deployment-smoke", "1",
    graph=lc.ModelGraph(
        source=lc.SourceSchema(), populations=(),
        fields=(lc.FieldSpec("amount", "actual"),),
        row_costs=(lc.RowCost("all", lc.field("amount")),),
        factors=lc.outer_factors() + (
            lc.Factor("payment", "payment", formula=lc.field("amount")),
        ),
        games=(lc.Game(
            "unit", lc.value("payment"), (lc.Switch("payment"),),
        ),),
        team_game="unit",
        organization=lc.organization_game("unit"),
        local=lc.local_game("unit"),
    ),
)
compiled = lc.compile_model(model)
display(lc.render_model(compiled))
rows = pd.DataFrame({
    "period": ["2026-08-01", "2026-09-01"],
    "employee_id": ["smoke-employee", "smoke-employee"],
    "employment_type": ["permanent", "permanent"],
    "team": ["Smoke", "Smoke"],
    "actual": [-100.0, 150.0],
})
dataset = lc.DatasetSnapshot("deployment-smoke", "1", rows)
state = lc.build_state(
    compiled, dataset, "2026-08", source_cost_column="actual",
)
assert abs(lc.cost(state, compiled) - (-100.0)) < 1e-6
result = lc.analyze(
    compiled, dataset,
    lc.AnalysisRequest(
        "2026-08", "2026-09", source_cost_column="actual",
        explainer="exact",
    ),
)
assert result.table("validation").passed.all()
assert abs(result.table("driver_bridge").impact.sum() - 250.0) < 1e-6
print("Source runtime smoke passed: -100 -> 150; reconciled change +250.")
display(result.table("driver_bridge"))

## Report existing results

This reads the calculated tables without another attribution run. Reports
require the reporting dependencies; turn off `enable_reporting` to check a
core-only environment. For restricted frontends, export a figure with
`charts.figures["organization"].write_html("payroll.html", include_plotlyjs=True)`.
Currency is a display label, not a currency inference or conversion.

In [ ]:
if enable_reporting:
    charts = lc.report(result, teams=["Smoke"], renderer=renderer)

## Retain provenance and portable results

`CODE.provenance` records the requested ref, commit, package version, archive
hash and file hashes, with no token. Retain it with deployment notes. It stays
separate from the existing analysis manifest; the loader does not change
bundle schemas. The optional cell below saves both a run and a neighboring
provenance file in a **new** directory.

To transfer results to another runtime, use its `lc.load_run(...)` on the
bundle. Do not pass Python model/result objects between handles.

In [ ]:
from pathlib import Path

print(json.dumps({
    key: value for key, value in CODE.provenance.items() if key != "files"
}, indent=2))
output_directory = None  # Example: "runs/source-runtime-demo"
if output_directory is not None:
    output = Path(output_directory)
    output.mkdir(parents=True, exist_ok=False)
    lc.export_run(result, output / "run")
    (output / "code-provenance.json").write_text(
        json.dumps(CODE.provenance, indent=2) + "\n", encoding="utf-8",
    )

## Finish or switch versions

Keep `CODE` open for the whole session. When finished, run `CODE.close()`;
it removes only this runtime's modules and temporary files and is safe to
call twice. Do not use retained `lc`, model, result or function references
after closing: cleanup cannot revoke existing Python references.

To update, finish/export your work, close the runtime, change `GITLAB_REF`,
and rerun from the loading cell, recreating all objects. A kernel restart
is an alternative way to start a fresh session. This isolates our package
code, not NumPy/SHAP or their shared process state; it does not enable
concurrent permutation execution.

The final cleanup cell is commented so **Run All leaves the example usable**.

In [ ]:
# Run explicitly when you have finished using this runtime:
# CODE.close()